# Demo TRM en Sudoku 🧩

Este notebook demuestra cómo usar la arquitectura TRM para resolver puzzles de Sudoku.

## Objetivos:
- Entrenar un modelo TRM en tareas de Sudoku
- Visualizar el proceso de razonamiento
- Comparar con modelos baseline
- Analizar la evolución de estados


In [ ]:
# Importar dependencias
import sys
import os
sys.path.append('..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Importar componentes TRM
from core.registry import create_model
from datasets.sudoku import create_sudoku_dataloaders, SudokuMetrics
from core.utils import get_device

# Configurar dispositivo
device = get_device()
print(f"Usando dispositivo: {device}")

# Configurar matplotlib
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12


## 1. Crear Dataset de Sudoku


In [ ]:
# Crear dataloaders de Sudoku
train_loader, val_loader, test_loader = create_sudoku_dataloaders(
    batch_size=32,
    train_size=1000,  # Dataset pequeño para demo
    val_size=200,
    test_size=200,
    difficulty='easy'
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# Mostrar ejemplo
for puzzles, solutions in train_loader:
    print(f"Puzzle shape: {puzzles.shape}")
    print(f"Solution shape: {solutions.shape}")
    
    # Mostrar primer ejemplo
    puzzle = puzzles[0].numpy()
    solution = solutions[0].numpy()
    
    print("\nPuzzle:")
    print(puzzle)
    print("\nSolución:")
    print(solution)
    break


## 2. Crear Modelo TRM


In [ ]:
# Crear modelo TRM para Sudoku
model_config = {
    'd_model': 256,
    'n_layers': 2,
    'n_inner': 6,
    'T': 3,
    'use_attention': False,
    'dropout': 0.1,
    'use_ema': True,
    'use_halting': True,
    'vocab_size': 10,  # 0-9 para Sudoku
    'max_seq_len': 81  # 9x9 grid
}

model = create_model('trm_base', model_config)
model = model.to(device)

print(f"Modelo creado con {sum(p.numel() for p in model.parameters() if p.requires_grad)} parámetros")
print(f"Configuración del modelo: {model.get_model_info()}")

# Probar forward pass
with torch.no_grad():
    sample_input = torch.randint(0, 10, (1, 81), device=device)
    output = model(sample_input)
    print(f"Input shape: {sample_input.shape}")
    print(f"Output shape: {output['output'].shape}")


## 3. Entrenamiento Rápido


In [ ]:
# Configurar entrenamiento
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# Métricas
metrics = SudokuMetrics()

# Entrenamiento rápido (solo unas pocas épocas para demo)
num_epochs = 5
train_losses = []
val_accuracies = []

model.train()
for epoch in range(num_epochs):
    # Entrenamiento
    epoch_loss = 0
    for batch_idx, (puzzles, solutions) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1}')):
        puzzles = puzzles.to(device)
        solutions = solutions.to(device)
        
        optimizer.zero_grad()
        output = model(puzzles)
        loss = criterion(output['output'].view(-1, 10), solutions.view(-1))
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Validación
    model.eval()
    val_acc = 0
    with torch.no_grad():
        for puzzles, solutions in val_loader:
            puzzles = puzzles.to(device)
            solutions = solutions.to(device)
            
            output = model(puzzles)
            predictions = output['output'].argmax(dim=-1)
            acc = metrics.accuracy(predictions, solutions)
            val_acc += acc
    
    val_acc /= len(val_loader)
    val_accuracies.append(val_acc)
    
    print(f'Epoch {epoch+1}: Loss={avg_loss:.4f}, Val Acc={val_acc:.4f}')
    model.train()

print("Entrenamiento completado!")


## 4. Visualizar Resultados


In [ ]:
# Visualizar curvas de entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Pérdida de entrenamiento
ax1.plot(train_losses, 'b-', label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Pérdida de Entrenamiento')
ax1.legend()
ax1.grid(True)

# Precisión de validación
ax2.plot(val_accuracies, 'r-', label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Precisión de Validación')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


## 5. Visualizar Proceso de Razonamiento


In [ ]:
# Obtener ejemplo de test
model.eval()
with torch.no_grad():
    for puzzles, solutions in test_loader:
        puzzle = puzzles[0:1].to(device)
        solution = solutions[0:1].to(device)
        
        # Forward pass con estados
        output = model(puzzle, return_states=True, return_halt_scores=True)
        prediction = output['output'].argmax(dim=-1)
        states = output.get('states', [])
        halt_scores = output.get('halt_scores', [])
        
        break

# Visualizar puzzle, solución y predicción
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

puzzle_np = puzzle[0].cpu().numpy().reshape(9, 9)
solution_np = solution[0].cpu().numpy().reshape(9, 9)
prediction_np = prediction[0].cpu().numpy().reshape(9, 9)

# Puzzle
im1 = axes[0].imshow(puzzle_np, cmap='viridis', vmin=0, vmax=9)
axes[0].set_title('Puzzle (Input)')
axes[0].axis('off')
plt.colorbar(im1, ax=axes[0])

# Solución
im2 = axes[1].imshow(solution_np, cmap='viridis', vmin=0, vmax=9)
axes[1].set_title('Solución (Target)')
axes[1].axis('off')
plt.colorbar(im2, ax=axes[1])

# Predicción
im3 = axes[2].imshow(prediction_np, cmap='viridis', vmin=0, vmax=9)
axes[2].set_title('Predicción')
axes[2].axis('off')
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

# Calcular métricas
accuracy = metrics.accuracy(prediction, solution)
grid_accuracy = metrics.grid_accuracy(prediction, solution)

print(f"Accuracy por celda: {accuracy:.4f}")
print(f"Accuracy por grid: {grid_accuracy:.4f}")


## 6. Análisis de Estados y Halting


In [ ]:
# Visualizar evolución de estados si están disponibles
if states:
    num_states = len(states)
    fig, axes = plt.subplots(2, num_states, figsize=(4 * num_states, 8))
    
    if num_states == 1:
        axes = axes.reshape(2, 1)
    
    for i, state in enumerate(states):
        # Estado
        state_np = state[0].cpu().numpy()  # Primer ejemplo del batch
        
        # Visualizar como heatmap
        sns.heatmap(state_np, ax=axes[0, i], cmap='viridis', cbar=True)
        axes[0, i].set_title(f'Estado - Iteración {i+1}')
        axes[0, i].set_xlabel('Dimensión')
        axes[0, i].set_ylabel('Posición')
    
    # Scores de parada
    if halt_scores:
        for i, halt in enumerate(halt_scores):
            if i < num_states:
                halt_np = halt[0].cpu().numpy()
                axes[1, i].plot(halt_np)
                axes[1, i].set_title(f'Scores de Parada - Iteración {i+1}')
                axes[1, i].set_xlabel('Posición')
                axes[1, i].set_ylabel('Score')
                axes[1, i].axhline(y=0.5, color='r', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()
else:
    print("No hay estados disponibles para visualizar")


## 7. Evaluación Final


In [ ]:
# Evaluación completa en test set
model.eval()
test_metrics = {
    'accuracy': 0,
    'grid_accuracy': 0,
    'row_accuracy': 0,
    'column_accuracy': 0,
    'box_accuracy': 0
}

with torch.no_grad():
    for puzzles, solutions in tqdm(test_loader, desc='Evaluando'):
        puzzles = puzzles.to(device)
        solutions = solutions.to(device)
        
        output = model(puzzles)
        predictions = output['output'].argmax(dim=-1)
        
        # Calcular métricas
        test_metrics['accuracy'] += metrics.accuracy(predictions, solutions)
        test_metrics['grid_accuracy'] += metrics.grid_accuracy(predictions, solutions)
        test_metrics['row_accuracy'] += metrics.row_accuracy(predictions, solutions)
        test_metrics['column_accuracy'] += metrics.column_accuracy(predictions, solutions)
        test_metrics['box_accuracy'] += metrics.box_accuracy(predictions, solutions)

# Promediar métricas
for metric in test_metrics:
    test_metrics[metric] /= len(test_loader)

# Mostrar resultados
print("=== RESULTADOS FINALES ===")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

# Visualizar distribución de accuracy por grid
grid_accuracies = []
with torch.no_grad():
    for puzzles, solutions in test_loader:
        puzzles = puzzles.to(device)
        solutions = solutions.to(device)
        
        output = model(puzzles)
        predictions = output['output'].argmax(dim=-1)
        
        # Calcular accuracy por grid
        for i in range(predictions.shape[0]):
            grid_acc = (predictions[i] == solutions[i]).all().float().item()
            grid_accuracies.append(grid_acc)

plt.figure(figsize=(10, 6))
plt.hist(grid_accuracies, bins=20, alpha=0.7, edgecolor='black')
plt.xlabel('Grid Accuracy')
plt.ylabel('Frecuencia')
plt.title('Distribución de Grid Accuracy en Test Set')
plt.axvline(np.mean(grid_accuracies), color='red', linestyle='--', label=f'Media: {np.mean(grid_accuracies):.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Porcentaje de grids resueltos correctamente: {np.mean(grid_accuracies)*100:.2f}%")


## 8. Conclusiones

Este demo ha mostrado:

1. **Creación de dataset**: Generación automática de puzzles de Sudoku
2. **Modelo TRM**: Implementación con estados persistentes y mecanismo de parada
3. **Entrenamiento**: Proceso de entrenamiento con métricas de seguimiento
4. **Visualización**: Análisis del proceso de razonamiento y evolución de estados
5. **Evaluación**: Métricas detalladas de rendimiento

### Próximos pasos:
- Experimentar con diferentes configuraciones de T y n_inner
- Comparar con modelos baseline (Transformer, LSTM)
- Probar en puzzles más difíciles
- Analizar el impacto del mecanismo de parada
